In [1]:
#importing libraries

import pandas as pd
import datetime as dt

In [12]:
#reading the file into df
#WARNING: set the proper path to the file on your local machine before running the code

df=pd.read_excel(r'C:\Users\Basil\Downloads\1631530784574520550test_task_data_manipulation.xlsx', sheet_name='RAW Data')
print(df.head())

         Incoming date            SKU  Internal Reference  \
0  2018-03-16 17:01:57  SKU0000001861              7320.0   
1  2018-07-12 19:19:28  SKU0000053109            212294.0   
2  2018-03-19 13:01:13  SKU0000040358            161608.0   
3  2018-03-20 13:06:39  SKU0000029576            118724.0   
4  2018-03-20 15:13:52  SKU0000040199            160972.0   

                                        Product Name    Condition  \
0  Used 205/55R16 Continental PureContact 91H - 6/32         used   
1  Used 285/35ZRF20 Bridgestone Potenza RE070R Ru...         used   
2  Driven Once 225/65R16 Falken Ziex ZE912 100H -...  driven_once   
3        Used 205/55R16 Definity HP 800 91H - 7.5/32         used   
4        Used 205/55R16 Michelin Defender 91T - 5/32         used   

         Brand                    Model         Size  Section  Aspect  ...  \
0  Continental              PureContact    205/55R16    205.0    55.0  ...   
1  Bridgestone  Potenza RE070R Run Flat  285/35ZRF20    285.0 

In [3]:
#checking data types & missing values

print(df.info())
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147865 entries, 0 to 147864
Data columns (total 31 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Incoming date                        147865 non-null  object 
 1   SKU                                  147865 non-null  object 
 2   Internal Reference                   147787 non-null  float64
 3   Product Name                         147865 non-null  object 
 4   Condition                            147726 non-null  object 
 5   Brand                                147726 non-null  object 
 6   Model                                147748 non-null  object 
 7   Size                                 147865 non-null  object 
 8   Section                              147864 non-null  float64
 9   Aspect                               147864 non-null  float64
 10  Rim                                  147864 non-null  float64
 11  Rad          

In [4]:
#determining the size of the result df

filtered_df=df[df['Sales Team'].notnull()]
df_length=10*filtered_df['Sales Team'].nunique()
print(df_length)

90


In [5]:
#top-10 for each team, checking the size

df1 = filtered_df[['Sales Team', 'Size', 'Qty (from quant)']].groupby(['Sales Team', 'Size']).agg({'Qty (from quant)':'sum'}).sort_values('Qty (from quant)', ascending=False)
result_df=df1.groupby('Sales Team').head(10).sort_values(['Sales Team','Qty (from quant)'], ascending=[True, False]).reset_index()
print(df_length==len(result_df))
print(result_df.head(20))

True
        Sales Team       Size  Qty (from quant)
0           Amazon  215/55R17              45.0
1           Amazon  205/55R16              41.0
2           Amazon  215/60R16              35.0
3           Amazon  225/65R17              33.0
4           Amazon  275/65R18              31.0
5           Amazon  235/60R18              29.0
6           Amazon  235/75R15              26.0
7           Amazon  215/70R15              26.0
8           Amazon  235/55R18              25.0
9           Amazon  265/70R17              25.0
10  Ars Innovation  205/55R16             334.0
11  Ars Innovation  215/55R17             206.0
12  Ars Innovation  265/60R18             147.0
13  Ars Innovation  225/65R17             125.0
14  Ars Innovation  235/65R18             119.0
15  Ars Innovation  235/55R20             117.0
16  Ars Innovation  275/55R20             114.0
17  Ars Innovation  235/55R18              93.0
18  Ars Innovation  265/70R17              92.0
19  Ars Innovation  225/45R17      

In [6]:
#counting the number of days between incoming and order

df['days_to_sell']=(pd.to_datetime(df['Sales Order (int) Date'])-pd.to_datetime(df['Incoming date'])).dt.days
print(df['days_to_sell'].head())

0    297.0
1    360.0
2    299.0
3    297.0
4    322.0
Name: days_to_sell, dtype: float64


In [7]:
#dealing with missing values, calculating average for each size/condition combo

filtered_df1=df[df['Sales Order (int) Date'].notnull() & df['Condition'].notnull()]
df_size_condition=filtered_df1[['Size', 'Condition', 'days_to_sell']].groupby(['Size', 'Condition']).agg({'days_to_sell':'mean'}).sort_values(['Size', 'days_to_sell']).reset_index()
print(df_size_condition.head(20))

           Size    Condition  days_to_sell
0     125/80D15         used         7.000
1     125/80D15          new        18.000
2   130/90R16MC         used        71.000
3     135/90D17         used       959.000
4     135/90R16         used        10.000
5   150/80R16MC         used        38.000
6     155/60R20         used        91.000
7     155/70D17  driven_once         3.000
8     155/80R13          new       250.875
9     155/80R13         used       291.000
10    155/80R13  driven_once       770.500
11    155/80R15          new         3.000
12    155/90D18         used        70.000
13       155R13          new       895.000
14    165/65R13         used       210.000
15    165/65R14          new         3.000
16    165/80R13  driven_once       127.000
17    165/80R13          new       283.000
18       165R15         used         7.000
19    175/55R15         used        62.000


In [8]:
#determining 1.25 times avg days_to_sell for each condition

df_25p=filtered_df1[['Condition', 'days_to_sell']].groupby(['Condition']).agg({'days_to_sell':'mean'})
df_25p['25p_more']=df_25p['days_to_sell']*1.25
df_25p=df_25p.reset_index()
df_25p=df_25p.drop('days_to_sell', 1)
print(df_25p)

     Condition    25p_more
0  driven_once  118.217938
1          new   82.466715
2      retread  149.202586
3         used   69.665849


C:\Users\Basil\AppData\Local\Temp\ipykernel_20912\2699361001.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df_25p=df_25p.drop('days_to_sell', 1)


In [9]:
#top-15 sizes by days_to_sell(given that the 1.25*avg condition is satisfied)

merged_df=df_size_condition.merge(df_25p, on='Condition')
filt_merged_df=merged_df[merged_df['days_to_sell']>merged_df['25p_more']].sort_values('days_to_sell').drop(['Condition','25p_more'], 1)
df_top15=filt_merged_df.drop_duplicates('Size').head(15).reset_index(drop=True)
print(df_top15)

           Size  days_to_sell
0    34X10.5R17     70.000000
1     155/90D18     70.000000
2     295/55R20     70.080000
3    255/55ZR18     70.333333
4    265/40ZR17     70.666667
5    225/40ZR19     70.972222
6   130/90R16MC     71.000000
7    285/40ZR19     71.714286
8     255/45R20     71.714953
9    295/30ZR21     72.000000
10    255/70R17     72.019084
11   225/50ZR18     72.026316
12   255/45ZR20     72.292683
13    225/60R17     72.444527
14    275/40R18     72.555556


C:\Users\Basil\AppData\Local\Temp\ipykernel_20912\2443414658.py:4: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  filt_merged_df=merged_df[merged_df['days_to_sell']>merged_df['25p_more']].sort_values('days_to_sell').drop(['Condition','25p_more'], 1)


In [10]:
#average prices for each size from the top-15 above

filtered_df3=filtered_df1[filtered_df1['Size'].isin(df_top15['Size'])]
prices=filtered_df3[['Size', 'Subtotal']].groupby('Size').agg({'Subtotal':'mean'}).reset_index()
prices_merged=df_top15.merge(prices, on='Size')
print(prices_merged)

           Size  days_to_sell    Subtotal
0    34X10.5R17     70.000000  140.400000
1     155/90D18     70.000000   47.990000
2     295/55R20     70.080000  133.102400
3    255/55ZR18     70.333333   73.012500
4    265/40ZR17     70.666667  104.893333
5    225/40ZR19     70.972222   81.771333
6   130/90R16MC     71.000000   46.990000
7    285/40ZR19     71.714286  102.010000
8     255/45R20     71.714953   84.109410
9    295/30ZR21     72.000000  142.140000
10    255/70R17     72.019084   59.254517
11   225/50ZR18     72.026316   57.478261
12   255/45ZR20     72.292683   91.171489
13    225/60R17     72.444527   52.741661
14    275/40R18     72.555556  125.964444


In [11]:
#uncomment, set the proper path and download if needed

#task 1
#result_df.to_csv(r'C:\Users\Basil\Documents\result_df.csv', index=False)
#result_df.to_excel(r'C:\Users\Basil\Documents\result_df.xlsx', index=False)

#task 2
#df_size_condition.to_csv(r'C:\Users\Basil\Documents\df_size_condition.csv', index=False)
#df_size_condition.to_excel(r'C:\Users\Basil\Documents\df_size_condition.xlsx', index=False)

#task 3&4
#prices_merged.to_csv(r'C:\Users\Basil\Documents\prices_merged.csv', index=False)
#prices_merged.to_excel(r'C:\Users\Basil\Documents\prices_merged.xlsx', index=False)